# Fine-tuning a pretrained text super-resolution model (TPGSR) on our whiteboard words

**Idea.** Our own small restorers only learned to wipe the blur next to strokes: they see 17 x 17 px and never learned what a letter is (see `sr_debug_activations.ipynb`). TPGSR is different:
- it was trained on **~17,000 real text pairs** (TextZoom), so it has seen many shapes of every character;
- its blocks contain **GRUs that read the whole row and column**, so every pixel sees the whole word, not a 17 px window;
- a frozen text reader (CRNN) tells it **which letters it is probably looking at** (the "text prior").

We fine-tune it on our marker handwriting and test whether it now helps PaddleOCR.

**How the data is made** (`scripts/finetune_tpgsr_whiteboard.py prepare`, already run once: 2,676 word pairs)
1. Every far/close **line** pair is cut into **words**. The cut positions come from the **far** photo only, as they would in the app.
2. Small input (LR) = far word, 16 x 64 px. Target (HR) = the same word in the close photo, 32 x 128 px (the TextZoom sizes).
3. Split: train = 17 Oct train photos + Airdrop synthetic pairs; validation = 17 Oct val photos + Airdrop B2 + 15% of the synthetic lines; **test = Airdrop B3, the CNN board, never used before**.

**Size rules** (from the word-swap test, `scripts/word_swap_counterfactual.py`). On B3, blur only costs PaddleOCR letters in words **under about 16-20 px** tall in the far photo; taller words read as well as the clear photo. TPGSR's output is 32 px tall, so a taller word can only *lose* detail on its route. So:
- training uses only words **10 to 32 px** tall (710 of 2,149: 507 synthetic Airdrop, 203 17 Oct). Under 10 px a person cannot read the word either (Lecture 8: an augmentation is valid only if a human keeps the label);
- validation uses only the lines that contain such a word (45 of 66), so it looks like the test board;
- at test time a **size gate** restores only words below a height chosen on validation (16 / 20 / 24 / 32 px); the rest stay raw.

**Training.** Starts from `outputs/tpgsr_textzoom/tpgsr_soft_prior_best.pt` (the team's TextZoom model), same loss as there (L1 + edge + soft text prior), learning rate 5e-5. Every batch is half our words, half TextZoom words, so it does not forget what it learned there. After every epoch the restored words are pasted back into their lines and **PaddleOCR reads the validation lines**; the epoch with the lowest CER is kept (epoch 0 = the pretrained model, so fine-tuning has to beat it).

**Test.** The same lines, read by PaddleOCR, 6 ways:

| Version | What it is |
|---|---|
| raw input | the far line as it is (should be about 0.153 on B3, as in the word-swap test: a sanity check) |
| 16px path, no network | words shrunk to 16 px and enlarged back with bicubic: the same route without the network |
| TPGSR pretrained | before fine-tuning |
| TPGSR fine-tuned | after fine-tuning (size gate applied, like every row below `raw input`) |
| TPGSR fine-tuned, every word | the same model with no size gate: shows what the gate saves |
| blend | fine-tuned output mixed with the original (weight chosen on validation) |
| gate | use the restored line only when PaddleOCR is more confident on it (margin chosen on validation) |

**Text prior check.** The prior comes from a frozen CRNN trained on printed scene text. If it misreads our blurry words, it tells TPGSR to draw the **wrong** letters clearly. Section 2 measures this before any training.

**Pass rule (fixed now, before training):** on the CNN board, the version chosen on validation must have a lower CER than **both** `raw input` and `16px path, no network`, **and** more lines must get better than worse.

In [1]:
import sys
from pathlib import Path
from IPython.display import Image, display


def find_repo_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for p in [start, *start.parents]:
        if (p / "requirements.txt").exists() and (p / "scripts").is_dir():
            return p
    raise FileNotFoundError("Open this notebook from inside the whiteboard-ocr repo folder")


REPO = find_repo_root()
SCRIPT = REPO / "scripts" / "finetune_tpgsr_whiteboard.py"
PY = sys.executable
for need in [REPO / "outputs" / "tpgsr_textzoom" / "tpgsr_soft_prior_best.pt", REPO.parent / "models" / "TPGSR",
             REPO / "data" / "textzoom" / "train1"]:
    print(("ok      " if need.exists() else "MISSING ") + str(need))

ok      <LOCAL_ASSIGNMENT>/whiteboard-ocr/outputs/tpgsr_textzoom/tpgsr_soft_prior_best.pt
ok      <LOCAL_ASSIGNMENT>/models/TPGSR
ok      <LOCAL_ASSIGNMENT>/whiteboard-ocr/data/textzoom/train1


## 1. Smoke test (2-4 min): does everything run?
A few lines, 2 tiny epochs, results in `outputs/tpgsr_whiteboard_smoke/` (the real outputs are not touched). Its numbers mean nothing; it only has to finish without an error.

In [2]:
!cd "{REPO}" && "{PY}" "{SCRIPT}" all --smoke

lines / words per split:
  test  17oct             2 lines     9 words
  test  airdrop_B3        4 lines     8 words
  train 17oct             6 lines    54 words
  train airdrop_synth     6 lines    13 words
  val   17oct             3 lines    30 words
  val   airdrop_synth     3 lines    12 words
saved 126 word pairs -> outputs/tpgsr_whiteboard_smoke
check the word splitting: outputs/tpgsr_whiteboard_smoke/word_split_preview.png
No Transformation module specified
No SequenceModeling module specified
CRNN (text prior source) reading our word crops. Word CER, lower = better; exact = whole word right.
Only lines whose label has one token per word box are used (word count = boxes).

=== Airdrop B3 (unseen board): 8 words from 4 of 4 lines ===
LR word (what the prior sees):   CER 0.656 | exact 0%
HR clear word (CRNN's ceiling):  CER 0.521 | exact 0%
worst LR readings: 'architecture'->'acblebien', 'cnn'->'cmy', 'cnn'->'cmy', 'cnn'->'cmy', 'architecture'->'achlehien'

=== Airdrop synthetic

## 2. Text prior check (about 1 min, no training)
How well does the CRNN behind the text prior read our word crops? It reads every LR word (what the prior sees) and its HR clear version (the best the CRNN can do on this handwriting). Words are matched to the label only when the line's label has one token per word box.
- **B3 LR CER under ~0.2:** the prior mostly names the right letters, go ahead.
- **Over ~0.4:** the prior is mostly noise. Fine-tuning can still learn to ignore it, but expect TPGSR to sharpen wrong letters; consider fine-tuning the CRNN on our words first.

In [3]:
!cd "{REPO}" && "{PY}" "{SCRIPT}" prior

No Transformation module specified
No SequenceModeling module specified
CRNN (text prior source) reading our word crops. Word CER, lower = better; exact = whole word right.
Only lines whose label has one token per word box are used (word count = boxes).

=== Airdrop B3 (unseen board): 50 words from 20 of 23 lines ===
LR word (what the prior sees):   CER 0.509 | exact 4%
HR clear word (CRNN's ceiling):  CER 0.258 | exact 28%
LR CER by size: < 16 px 0.463 (22 words) | >= 16 px 0.545 (28 words)
worst LR readings: 'via'->'he', 'feature'->'invve', 'feature'->'htn', 'filter'->'mkr', 'architecture'->'aestghees'

=== Airdrop synthetic, validation: 44 words from 16 of 40 lines ===
LR word (what the prior sees):   CER 0.327 | exact 39%
HR clear word (CRNN's ceiling):  CER 0.146 | exact 80%
LR CER by size: < 16 px 0.452 (18 words) | >= 16 px 0.240 (26 words)
worst LR readings: '1'->'i', 'vs'->'a', '1'->'i', 'vs'->'a', '1'->'i'

=== 17 Oct held-out photos: 74 words from 18 of 40 lines ===
LR word 

## 3. The real run (about 15-25 min on a Mac)
The word pairs are already prepared (`outputs/tpgsr_whiteboard/`), so this runs `train` (up to 10 epochs, stops early) -> `evaluate`. Run `prepare` first only if the paired datasets changed. To only re-test a trained model, run just the `evaluate` part.

In [4]:
!cd "{REPO}" && "{PY}" "{SCRIPT}" train && "{PY}" "{SCRIPT}" evaluate

No Transformation module specified
No SequenceModeling module specified
start: outputs/tpgsr_textzoom/tpgsr_soft_prior_best.pt (TextZoom epoch 9) on mps
training words: 710 of 2149 kept (10 <= box < 32 px in the far photo; 94 too small to read, 1345 too big to need SR)
  17oct: 203 words
  airdrop_synth: 507 words
<LOCAL_ASSIGNMENT>/.venv-paddle/lib/python3.12/site-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `<LOCAL_HOME>/.paddlex/official_models/PP-OCRv6_medium_rec`.
validation: 45 of 66 lines (those with a word < 32 px) | raw input CER 0.160 | epoch 0 (pretrained) CER 0.196
epoch  1: loss 0.0792 | val CER 0.204 (raw 0.160) | 114 s
epoch  2: l

## 4. Results

In [5]:
import json
import matplotlib.pyplot as plt
import pandas as pd

OUT = REPO / "outputs" / "tpgsr_whiteboard"
print((OUT / "summary.txt").read_text())

h = pd.DataFrame(json.load(open(OUT / "training_history.json")))
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
ax[0].plot(h.epoch, h.train_loss, marker="o"); ax[0].set_title("training loss")
ax[1].plot(h.epoch, h.val_cer, marker="o", label="fine-tuned"); ax[1].axhline(h.val_cer.iloc[0], ls="--", c="grey", label="epoch 0 = pretrained")
ax[1].set_title("validation CER from PaddleOCR (lower = better)"); ax[1].legend()
for a in ax:
    a.set_xlabel("epoch"); a.grid(alpha=0.3)
plt.tight_layout(); plt.show()

Size gate chosen on validation: only words < 16 px (far photo) are restored, in every version except 'raw input' and 'TPGSR fine-tuned, every word'. The 16px path uses the same gate, so it isolates the network.

=== Airdrop B3 (unseen board, main test): 23 lines ===
raw input                  CER 0.153 | vs raw +0.000 [+0.000, +0.000] | better 0 / same 23 / worse 0
16px path, no network      CER 0.153 | vs raw -0.001 [-0.016, +0.016] | better 2 / same 19 / worse 2
TPGSR pretrained           CER 0.166 | vs raw -0.014 [-0.040, +0.004] | better 1 / same 18 / worse 4
TPGSR fine-tuned           CER 0.166 | vs raw -0.014 [-0.040, +0.004] | better 1 / same 18 / worse 4
TPGSR fine-tuned, every word CER 0.247 | vs raw -0.094 [-0.136, -0.059] | better 2 / same 7 / worse 14
blend (alpha 0.3)          CER 0.159 | vs raw -0.007 [-0.025, +0.016] | better 3 / same 15 / worse 5
gate (margin 0.0)          CER 0.167 | vs raw -0.015 [-0.024, -0.005] | better 1 / same 18 / worse 4

=== 17 Oct held-out pho

<Figure size 1200x350 with 2 Axes>

In [6]:
df = pd.read_csv(OUT / "ocr_comparison.csv")
main = df[df.set.str.startswith("Airdrop B3")]
table = main.pivot_table(index="label", columns="method", values="cer", aggfunc="mean")
display(table.round(3))          # per line text: CER of every version
display(Image(filename=str(OUT / "word_split_preview.png")))
display(Image(filename=str(OUT / "visual_comparison.png")))

method,"16px path, no network",TPGSR fine-tuned,"TPGSR fine-tuned, every word",TPGSR pretrained,blend (alpha 0.3),gate (margin 0.0),raw input
label,,,,,,,
CNN Architecture,0.117,0.117,0.344,0.117,0.117,0.117,0.117
"LeNet, Alexnet, Resnet",0.227,0.227,0.227,0.227,0.182,0.182,0.227
Saliency via occlusion,0.250,0.386,0.386,0.386,0.273,0.341,0.273
Visualise feature layer,0.217,0.246,0.246,0.246,0.217,0.232,0.217
Visualise filter & output,0.170,0.170,0.180,0.170,0.170,0.170,0.170
Visualising CNN,0.107,0.093,0.093,0.093,0.133,0.133,0.093


<IPython.core.display.Image object>

<IPython.core.display.Image object>

## 5. How to read it
- **Sanity check:** `raw input` on B3 should be about 0.153 (the word-swap test; runs 1-4, M1 and M2 got 0.157 on 24 lines, one of which has no word box here). If it is far off, the lines or the CER differ from the earlier tests and the numbers are not comparable.
- **`16px path, no network` vs `raw input`:** what we lose just by shrinking words to the TextZoom size. If this is already much worse than raw, the model starts with less detail than PaddleOCR had.
- **`TPGSR pretrained` vs `fine-tuned`:** did our handwriting help?
- **`fine-tuned` vs `fine-tuned, every word`:** on B3 every word is under 32 px, so these match unless the gate chose 16-24 px. On 17 Oct the gated row should equal `raw input` almost everywhere (its words are big): that is the "does no harm" check.
- **The prior check (section 2):** if the B3 LR CER was high and fine-tuning still fails, the wrong prior is the first suspect, not the receptive field.
- **The verdict line** decides. Write it in the report whatever it says.